<a href="https://colab.research.google.com/github/chivian/Natural_Language_Processing/blob/master/nlp_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Note: We will start by setting up our environment and loading a very small slice of a dataset. Fine-tuning is computationally heavy, so we are only using 500 rows to demonstrate the mechanics quickly.

In [ ]:
# Cell 1: Setup and Data Loading
!pip install transformers[torch] datasets -q

import pandas as pd
from datasets import load_dataset

# Load a small slice of the 'xsum' (Extreme Summarisation) dataset for speed
print("Downloading dataset slice...")
dataset = load_dataset("xsum", split="train[:500]")

# Split into a tiny train and validation set
dataset = dataset.train_test_split(test_size=0.1)

print("--- Data Loading Complete ---")
print(f"Training examples: {len(dataset['train'])}")
print(f"Validation examples: {len(dataset['test'])}")
print("\nSample Document:\n", dataset['train'][0]['document'][:200], "...")
print("\nSample Summary:\n", dataset['train'][0]['summary'])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/300M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/204045 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11332 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11334 [00:00<?, ? examples/s]

--- Data Loading Complete ---
Training examples: 450
Validation examples: 50

Sample Document:
 Allen scored his first international goal on his 32nd appearance in Wales' 4-0 win over Moldova in their opening 2018 World Cup qualifier.
The 26-year-old joined Stoke City from Liverpool for £13m in  ...

Sample Summary:
 Wales midfielder Joe Allen says he feels "refreshed and positive" following his summer move from Liverpool to Stoke City.


Note: Seq2Seq tokenisation is slightly more complex than standard classification. We must tokenise the input document as standard, but we must also tokenise the target summary and assign it to a special variable called labels.

In [ ]:
# Cell 2: Dual Tokenisation for Seq2Seq
from transformers import AutoTokenizer

# Initialise the tokeniser for T5
model_checkpoint = "t5-small"
tokeniser = AutoTokenizer.from_pretrained(model_checkpoint)

# The T5 model requires a specific prefix to know what task it is doing
prefix = "summarize: "

def preprocess_function(examples):
    # Add the prefix to all inputs
    inputs = [prefix + doc for doc in examples["document"]]

    # Tokenise the inputs
    model_inputs = tokeniser(inputs, max_length=512, truncation=True)

    # Tokenise the target summaries (labels)
    # text_target is specifically used for the labels in newer Hugging Face versions
    labels = tokeniser(text_target=examples["summary"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the tokenisation to the entire dataset
print("Tokenising datasets...")
tokenised_datasets = dataset.map(preprocess_function, batched=True)
print("Tokenisation complete!")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenising datasets...


Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Tokenisation complete!


Note: Now we configure the Hugging Face Trainer. We define our hyperparameters (like batch size and epochs). Even with 500 rows, this step may take a few minutes if you are using a standard CPU, or seconds if you have enabled a GPU.

In [ ]:
# Cell 3: Model Initialisation and Training Setup
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

# Initialise the pre-trained T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

# Initialise the Data Collator to handle dynamic padding for our batches
data_collator = DataCollatorForSeq2Seq(tokenizer=tokeniser, model=model)

# Define the training hyperparameters
training_args = Seq2SeqTrainingArguments(
    output_dir="./summarisation_model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
)

# Initialise the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenised_datasets["train"],
    eval_dataset=tokenised_datasets["test"],
    processing_class=tokeniser,
    data_collator=data_collator,
)

# Execute the fine-tuning loop
print("Beginning fine-tuning process. This may take a moment...")
trainer.train()
print("Fine-tuning complete!")

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Beginning fine-tuning process. This may take a moment...


Epoch,Training Loss,Validation Loss
1,No log,2.999050
2,No log,2.823039
3,No log,2.786056


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuning complete!


Note: Let us put our newly fine-tuned model to the test! We will pass it a completely unseen block of text and ask it to generate a summary.

In [ ]:
# Cell 4: Inference (Testing the Fine-Tuned Model natively)
import torch

unseen_text = """
The rapid advancements in artificial intelligence over the past decade have fundamentally
shifted how software engineers approach problem-solving. While traditional programming required
developers to write explicit, rules-based logic to handle every conceivable edge case, modern
machine learning allows systems to infer rules directly from massive datasets. This paradigm shift
has led to breakthroughs in natural language processing, computer vision, and autonomous robotics.
However, it has also introduced new challenges, such as the need for vast amounts of computational
power, concerns regarding algorithmic bias, and the complex task of interpreting "black box"
neural networks. Despite these hurdles, the integration of AI into everyday applications continues
to accelerate at an unprecedented pace.
"""

# 1. Prepare the input with the T5 prefix
t5_input = "summarize: " + unseen_text

# 2. Tokenise the input (convert human text to a PyTorch tensor of numbers)
# return_tensors="pt" tells the tokeniser to format it specifically for PyTorch
input_ids = tokeniser(t5_input, return_tensors="pt").input_ids

# Ensure the inputs are on the same hardware device as the model (CPU or GPU)
input_ids = input_ids.to(model.device)

# 3. Generate the summary tokens using the model's native generate function
print("Generating summary...\n")
summary_ids = model.generate(
    input_ids,
    min_length=10,
    max_new_tokens=50,
    num_beams=4, # Uses beam search to explore multiple paths for a better summary
    early_stopping=True
)

# 4. Decode the generated tokens back into human-readable text
# skip_special_tokens=True removes structural tags like <pad> or </s>
summary_text = tokeniser.decode(summary_ids[0], skip_special_tokens=True)

print("--- Final Generated Summary ---")
print(summary_text)

Generating summary...

--- Final Generated Summary ---
the rapid advancements in artificial intelligence have fundamentally shifted how software engineers approach problem-solving.
